# XGBoost

In [37]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
from datetime import timedelta
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}

# Parametri

In [38]:
def make_rskf(y_strat, n_repeats=3, max_splits=5):
    
    min_class = y_strat.value_counts().min()
    n_splits = min(max_splits, min_class)

    if n_splits < 2:
        return None

    return RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42
    )

# Training Duke
 

In [39]:
# ============================================================
# TRAINING DUKE (GRID PER TARGET)
# ============================================================

def training_duke(file_path, csv_name):

    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID'")

    # =======================
    # TARGET
    # =======================
    df["ER_class"]   = pd.to_numeric(df["ER"], errors="coerce")
    df["PR_class"]   = pd.to_numeric(df["PR"], errors="coerce")

    targets = ["ER_class", "PR_class"]
    df = df.dropna(subset=targets).copy()
    for c in targets:
        df[c] = df[c].astype(int)

    targets = [c for c in targets if df[c].nunique() > 1]
    if len(targets) == 0:
        return None

    # =======================
    # FEATURES
    # =======================
    drop_cols = [
        "Patient ID","lesion idx","tumor/benign",
        "GRADE","isTN","Breast",
        "ER","PR","HER2"
    ] + targets

    X = df.drop(columns=drop_cols, errors="ignore")
    y = df[targets]

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    # =======================
    # CV STRATIFICATA (ER + PR + HER2 WRAP)
    # =======================
    y_strat = (
        y["ER_class"].astype(str) + "_" +
        y["PR_class"].astype(str) + "_" 
    )

    print("\nDistribuzione classi combinate (ER_PR_HER2):")
    print(y_strat.value_counts())

    n_splits = min(5, y_strat.value_counts().min())


    if n_splits < 2:
        return None

    cv = make_rskf(y_strat, n_repeats=3, max_splits=5)
    if cv is None:
        return None

    splits = list(cv.split(X, y_strat))



    # =======================
    # GRID
    # =======================
    param_grid = {
        "n_estimators": [50, 75, 100],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.05, 0.1, 0.2]
    }

    fold_reports = []

    # =======================
    # CV LOOP
    # =======================
    for fold_id, (tr, te) in enumerate(splits):

        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        fold_metrics = {}

        for col in targets:

            pos = y_tr[col].sum()
            neg = len(y_tr) - pos
            spw = neg / max(pos,1)

            base_model = XGBClassifier(
                random_state=42,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                scale_pos_weight=spw,
                n_jobs=1
            )

            grid = GridSearchCV(
                base_model,
                param_grid=param_grid,
                scoring="f1_macro",
                cv=3,
                n_jobs=-1
            )

            grid.fit(X_tr, y_tr[col])
            best_model = grid.best_estimator_

            
            y_prob = best_model.predict_proba(X_te)[:,1]
            y_pred = (y_prob >= 0.4).astype(int)
            
            # ===== STAMPE =====
            print(f"\nTarget: {col}")
            print("Confusion Matrix")
            print(confusion_matrix(y_te[col], y_pred))
            print("Classification Report")
            print(classification_report(
                y_te[col],
                y_pred,
                zero_division=0
            ))

            fold_metrics[col] = {
                "f1": f1_score(y_te[col], y_pred, average="macro", zero_division=0),
                "accuracy": accuracy_score(y_te[col], y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_te[col], y_pred),
                "auc": roc_auc_score(y_te[col], y_prob)
            }

        fold_reports.append(fold_metrics)

    return {
        "fold_reports": fold_reports,
        "targets_used": targets
    }


# Training ambl

In [40]:
# ============================================================
# TRAINING AMBL (GRID PER TARGET)
# ============================================================

def training_ambl(file_path, csv_name):

    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID'")

    # =======================
    # TARGET
    # =======================
    df["ER_class"] = (
        pd.to_numeric(df["ER [SII]"], errors="coerce") >= 1
    ).astype(int)

    df["PR_class"] = (
        pd.to_numeric(df["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

    targets = ["ER_class", "PR_class"]
    df = df.dropna(subset=targets).copy()

    targets = [c for c in targets if df[c].nunique() > 1]
    if len(targets) == 0:
        return None

    # =======================
    # FEATURES
    # =======================
    drop_cols = [
        "Patient ID","lesion idx","tumor/benign",
        "GRADE","isTN","Breast",
        "ER","PR",
        "ER [SII]","PR [SII]"
    ] + targets

    X = df.drop(columns=drop_cols, errors="ignore")
    y = df[targets]

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    # =======================
    # CV STRATIFICATA (ER + PR + HER2 WRAP)
    # =======================
    y_strat = (
        y["ER_class"].astype(str) + "_" +
        y["PR_class"].astype(str) + "_"
    )

    print("\nDistribuzione classi combinate (ER_PR_HER2):")
    print(y_strat.value_counts())

    n_splits = min(5, y_strat.value_counts().min())


    if n_splits < 2:
        return None

    cv = make_rskf(y_strat, n_repeats=5, max_splits=2)
    if cv is None:
        return None

    splits = list(cv.split(X, y_strat))



    # =======================
    # GRID
    # =======================
    param_grid = {
        "n_estimators": [50, 75, 100],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.05, 0.1, 0.2]
    }

    fold_reports = []

    # =======================
    # CV LOOP
    # =======================
    for fold_id, (tr, te) in enumerate(splits):

        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        fold_metrics = {}

        for col in targets:

            pos = y_tr[col].sum()
            neg = len(y_tr) - pos
            spw = neg / max(pos,1)

            base_model = XGBClassifier(
                random_state=42,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                scale_pos_weight=spw,
                n_jobs=1
            )

            grid = GridSearchCV(
                base_model,
                param_grid=param_grid,
                scoring="f1_macro",
                cv=3,
                n_jobs=-1
            )

            grid.fit(X_tr, y_tr[col])
            best_model = grid.best_estimator_

            
            y_prob = best_model.predict_proba(X_te)[:,1]
            y_pred = (y_prob >= 0.4).astype(int)

            print(f"\nTarget: {col}")
            print("Confusion Matrix")
            print(confusion_matrix(y_te[col], y_pred))
            print("Classification Report")
            print(classification_report(
                y_te[col],
                y_pred,
                zero_division=0
            ))

            fold_metrics[col] = {
                "f1": f1_score(y_te[col], y_pred, average="macro", zero_division=0),
                "accuracy": accuracy_score(y_te[col], y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_te[col], y_pred),
                "auc": roc_auc_score(y_te[col], y_prob)
            }

        fold_reports.append(fold_metrics)

    return {
        "fold_reports": fold_reports,
        "targets_used": targets
    }


# Vado a stampare il risultato in un formato leggibile

In [41]:
def print_grid_search_results(results_per_dataset, save_csv=True, output_path="XGBoost.csv"):
    print("\n" + "=" * 80)
    print(" " * 20 + "Metriche (MEDIA ± STD) per target")
    print("=" * 80)

    rows = []

    for dataset_name, best_result in results_per_dataset.items():
        if best_result is None:
            continue

        fold_reports = best_result["fold_reports"]
        target_names = best_result.get("targets_used", [])

        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        for target_name in target_names:

            f1_list  = np.array([fold[target_name]["f1"] for fold in fold_reports], dtype=float)
            acc_list = np.array([fold[target_name]["accuracy"] for fold in fold_reports], dtype=float)
            auc_list = np.array([fold[target_name]["auc"] for fold in fold_reports], dtype=float)
            bal_list = np.array([fold[target_name]["balanced_accuracy"] for fold in fold_reports], dtype=float)

            f1_mean,  f1_std  = np.mean(f1_list),  np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
            bal_mean, bal_std = np.mean(bal_list), np.std(bal_list)

            valid_auc = ~np.isnan(auc_list)
            auc_mean = np.mean(auc_list[valid_auc]) if valid_auc.any() else np.nan
            auc_std  = np.std(auc_list[valid_auc])  if valid_auc.any() else np.nan

            # ===== STAMPA =====
            print(f"\nTarget: {target_name}")
            print(f"  F1-score           = {f1_mean:.3f}  ±  {f1_std:.3f}")
            print(f"  Accuracy           = {acc_mean:.3f}  ±  {acc_std:.3f}")
            print(f"  Balanced Accuracy  = {bal_mean:.3f}  ±  {bal_std:.3f}")
            print(
                f"  AUC                = {auc_mean:.3f}  ±  {auc_std:.3f}"
                if not np.isnan(auc_mean)
                else f"  AUC                = NaN     ±  NaN"
            )

            # ===== CSV =====
            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "F1-score": f"{f1_mean:.3f} ± {f1_std:.3f}",
                "Accuracy": f"{acc_mean:.3f} ± {acc_std:.3f}",
                "Balanced Accuracy": f"{bal_mean:.3f} ± {bal_std:.3f}",
                "AUC": (
                    f"{auc_mean:.3f} ± {auc_std:.3f}"
                    if not np.isnan(auc_mean)
                    else "NaN ± NaN"
                )
            })

    # ===== SALVATAGGIO FILE =====
    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        output_path = Path(output_path)
        df_out.to_csv(output_path, index=False)
        print(f"\n Risultati salvati in: {output_path.resolve()}")


# Lettura dei file

In [42]:
start_time = time.time()

# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():

    name_lower = name.lower()

    if "ambl" in name_lower:
        print(f"\n>>> Training AMBL: {name}")
        results_per_dataset[name] = training_ambl(file_path, name)
        print_grid_search_results(results_per_dataset)
    elif "duke" in name_lower:
        print(f"\n>>> Training DUKE: {name}")
        results_per_dataset[name] = training_duke(file_path, name)
        print_grid_search_results(results_per_dataset)
    else:   
        raise ValueError(f"Dataset non riconosciuto: {name}")
    
end_time = time.time()

# Tempo totale
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



>>> Training DUKE: duke_lesions

Distribuzione classi combinate (ER_PR_HER2):
1_1_    126
0_0_    115
1_0_     42
0_1_      8
Name: count, dtype: int64

Target: ER_class
Confusion Matrix
[[ 4 21]
 [ 5 29]]
Classification Report
              precision    recall  f1-score   support

           0       0.44      0.16      0.24        25
           1       0.58      0.85      0.69        34

    accuracy                           0.56        59
   macro avg       0.51      0.51      0.46        59
weighted avg       0.52      0.56      0.50        59


Target: PR_class
Confusion Matrix
[[ 8 24]
 [ 6 21]]
Classification Report
              precision    recall  f1-score   support

           0       0.57      0.25      0.35        32
           1       0.47      0.78      0.58        27

    accuracy                           0.49        59
   macro avg       0.52      0.51      0.47        59
weighted avg       0.52      0.49      0.46        59


Target: ER_class
Confusion Matrix
[[ 8 1